In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs MDAnalysis and sets up a sample trajectory
# using MDAnalysis's built-in test data (adenylate kinase, GROMACS format).
#
# This replaces the "peptide.tpr / peptide.xtc" files that would be used in a
# local setup with your own simulation data. The built-in AdK trajectory covers
# the same analysis concepts (RMSD, RMSF, Rg, hydrogen bonds, etc.).
#
# numpy and matplotlib are pre-installed in Google Colab.
# MDAnalysis installation takes ~60 seconds on first run.

import sys, os, subprocess, shutil, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# ── Install MDAnalysis ────────────────────────────────────────────────────────
print("\nInstalling MDAnalysis (this may take ~60 s)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "MDAnalysis", "MDAnalysisTests"])
print("MDAnalysis installed ✓")

# ── Set up built-in test trajectory ──────────────────────────────────────────
# MDAnalysis ships with a 10-frame GROMACS trajectory of adenylate kinase (AdK)
# solved in TIP4P water with the OPLS/AA force field.  It is a fully realistic
# MD system and exercises exactly the same analysis code as a peptide simulation.
from MDAnalysis.tests.datafiles import GRO as _TOPOLOGY_SRC, XTC as _TRAJ_SRC

shutil.copy(_TOPOLOGY_SRC, "peptide.gro")   # topology  (GRO format)
shutil.copy(_TRAJ_SRC,     "peptide.xtc")   # trajectory (XTC format)

print(f"Topology   → peptide.gro  (source: {os.path.basename(_TOPOLOGY_SRC)})")
print(f"Trajectory → peptide.xtc  (source: {os.path.basename(_TRAJ_SRC)})")
print("\nNote: Using adenylate kinase (AdK) as the example system.")
print("To analyse your own simulation, replace peptide.gro / peptide.xtc")
print("with your own topology and trajectory files.")
print("\nEnvironment ready ✓")


# Introduction to MDAnalysis: Analyzing Peptide Simulations

**Prerequisites:** This notebook assumes you have completed *Part 1: Core Concepts* and have a working knowledge of Python variables, lists, loops, functions, and file I/O.

**What is MDAnalysis?**  
MDAnalysis is a Python library that lets you read, manipulate, and analyze the output of molecular dynamics (MD) simulations. It supports many popular file formats, including those produced by GROMACS (`.tpr`, `.xtc`, `.gro`), NAMD (`.psf`, `.dcd`), AMBER (`.prmtop`, `.nc`), and many others. You can think of it as a Swiss army knife for MD trajectories.

**Files you will need for this notebook:**

| File | Role | Description |
|------|------|-------------|
| `peptide.tpr` | Topology | GROMACS run-input file — contains all atom definitions, bonds, charges, and force-field parameters |
| `peptide.xtc` | Trajectory | Compressed GROMACS trajectory — stores atom positions at each saved timestep |

Place both files in the same directory as this notebook before running the cells.

### Topics covered

| Section | Topic | Key MDAnalysis concept |
|---------|-------|------------------------|
| 1 | Installation & imports | `import MDAnalysis` |
| 2 | Loading a simulation | `Universe` |
| 3 | Selecting atoms | `AtomGroup`, selection language |
| 4 | Inspecting the system | Atom attributes (`names`, `resnames`, `positions`) |
| 5 | Iterating over frames | The trajectory loop |
| 6 | Center of mass & radius of gyration | Per-frame scalar analysis |
| 7 | RMSD: measuring structural drift | `MDAnalysis.analysis.rms` |
| 8 | RMSF: per-residue flexibility | `MDAnalysis.analysis.rms.RMSF` |
| 9 | Distances and contacts | `MDAnalysis.lib.distances` |
| 10 | Hydrogen bond analysis | `MDAnalysis.analysis.hydrogenbonds` |
| 11 | Writing a reusable analysis script | Putting it all together |

> **How to use this notebook:** Read each explanation, run the **Example** cell with **Shift + Enter**, then complete the **✏️ Practice** cell on your own.  
> Each practice cell contains variable stubs already named for you — replace each `None` with the correct value, then delete the `raise NotImplementedError()` line.  
> Run the **Test** cell immediately after to check your work: ✓ means you passed.

---
## Section 1 — Installation and Imports

MDAnalysis is not part of the Python standard library, so it must be installed separately. The recommended way is through `conda` or `pip`.

```bash
# Using conda (recommended — handles all compiled dependencies)
conda install -c conda-forge mdanalysis mdanalysistests

# Or using pip
pip install MDAnalysis MDAnalysisTests
```

The cell below checks that MDAnalysis is installed and prints the version.

In [ ]:
import MDAnalysis as mda
print("MDAnalysis version:", mda.__version__)

# Other packages used throughout this notebook
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# MDAnalysis analysis submodules (imported as needed later)
from MDAnalysis.analysis import rms, align
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
import MDAnalysis.lib.distances as distances

print("All imports successful!")

---
## Section 2 — Loading a Simulation: The `Universe`

The central object in MDAnalysis is the **`Universe`**. A `Universe` holds *everything* about a simulation:
- the **topology** (atom names, residue names, bonds, charges, masses)
- the **trajectory** (positions as a function of time)

You create a `Universe` by passing the topology file and trajectory file to `mda.Universe()`:

```python
u = mda.Universe("topology_file", "trajectory_file")
```

For GROMACS data, the topology is a `.tpr` file and the trajectory is an `.xtc` file.

### A note on coordinate units
MDAnalysis stores all distances in **Ångströms (Å)** and all times in **picoseconds (ps)**, regardless of what the simulation program used internally. You do not need to perform unit conversions for the quantities you will encounter in this notebook.

In [ ]:
# --- EXAMPLE ---
# Load the topology and trajectory into a Universe.
# Adjust the filenames below to match the files you have placed alongside this notebook.

TPR = "peptide.gro"   # GROMACS topology (GRO format — loaded from MDAnalysis test data in setup cell)
XTC = "peptide.xtc"   # GROMACS trajectory

u = mda.Universe(TPR, XTC)

print(f"Number of atoms   : {u.atoms.n_atoms}")
print(f"Number of residues: {u.atoms.n_residues}")
print(f"Number of frames  : {len(u.trajectory)}")
print(f"Timestep (ps)     : {u.trajectory.dt:.2f}")
print(f"Total time (ns)   : {u.trajectory.totaltime / 1000:.3f}")

#### ✏️ Practice 2.1
Use the `Universe` object `u` to fill in the variables below.

- `n_atoms` — the total number of atoms
- `n_residues` — the number of residues
- `sim_length_ns` — the total simulation length **in nanoseconds** (hint: `u.trajectory.totaltime` is in ps; 1 ns = 1000 ps)

In [ ]:
# ✏️ YOUR CODE HERE
n_atoms       = None   # total atoms in the system
n_residues    = None   # total residues
sim_length_ns = None   # total simulation time in nanoseconds

raise NotImplementedError()

In [ ]:
# TEST 2.1
assert isinstance(n_atoms, int) and n_atoms > 0, "n_atoms should be a positive integer"
assert isinstance(n_residues, int) and n_residues > 0, "n_residues should be a positive integer"
assert sim_length_ns > 0, "sim_length_ns should be positive"
assert abs(sim_length_ns - u.trajectory.totaltime / 1000) < 1e-6, "sim_length_ns conversion is off"
print(f"✓  {n_atoms} atoms, {n_residues} residues, {sim_length_ns:.3f} ns simulation")

---
## Section 3 — Selecting Atoms

A `Universe` contains all atoms in the system — solute, solvent, ions, and everything else. Most analyses focus on a **subset** of atoms. MDAnalysis provides a human-readable **selection language** that should feel familiar if you have ever used VMD or CHARMM.

You call `u.select_atoms("selection string")` and get back an **`AtomGroup`** — an ordered collection of atoms on which you can compute properties.

### Common selection keywords

| Keyword | Selects | Example |
|---------|---------|--------|
| `protein` | All protein atoms | `"protein"` |
| `backbone` | N, CA, C, O backbone atoms | `"backbone"` |
| `resname ABC` | Residues named ABC | `"resname ALA"` |
| `resid 1:10` | Residue numbers 1–10 | `"resid 1:10"` |
| `name CA` | Atoms named CA | `"name CA"` |
| `not water` | Everything except water | `"not resname SOL"` |
| `and`, `or`, `not` | Boolean logic | `"protein and name CA"` |
| `around N sel` | Atoms within N Å of sel | `"around 5.0 resname LIG"` |

In [ ]:
# --- EXAMPLE ---
# Select different subsets of the system

protein   = u.select_atoms("protein")
backbone  = u.select_atoms("backbone")
ca_atoms  = u.select_atoms("protein and name CA")
solvent   = u.select_atoms("resname SOL")   # GROMACS water residue name

print(f"Protein atoms  : {protein.n_atoms}")
print(f"Backbone atoms : {backbone.n_atoms}")
print(f"Cα atoms       : {ca_atoms.n_atoms}  (one per residue)")
print(f"Solvent atoms  : {solvent.n_atoms}")
print()
print("First 5 Cα residue names:", ca_atoms.resnames[:5])

#### ✏️ Practice 3.1
Create the following `AtomGroup` objects:
- `heavy_protein` — all protein atoms that are **not hydrogen** (hint: hydrogen atom names start with `H`; use `not name H*`)
- `first_five` — Cα atoms of the **first five residues** (hint: `resid 1:5` and `name CA`)

In [ ]:
# ✏️ YOUR CODE HERE
heavy_protein = None
first_five    = None

raise NotImplementedError()

In [ ]:
# TEST 3.1
import MDAnalysis.core.groups
assert isinstance(heavy_protein, MDAnalysis.core.groups.AtomGroup), "heavy_protein should be an AtomGroup"
assert all('H' not in name[0] for name in heavy_protein.names), "heavy_protein should contain no H atoms"
assert isinstance(first_five, MDAnalysis.core.groups.AtomGroup), "first_five should be an AtomGroup"
assert first_five.n_atoms == 5, f"Expected 5 Cα atoms, got {first_five.n_atoms}"
assert list(first_five.resids) == list(range(1, 6)), "first_five should cover resids 1–5"
print("✓  Selections look correct")

---
## Section 4 — Inspecting the System

An `AtomGroup` exposes atom-level information as **numpy arrays**. Since you learned about lists and loops in Part 1, numpy arrays should feel intuitive — they behave like lists but support fast element-wise math.

### Useful `AtomGroup` attributes

| Attribute | Type | Contents |
|-----------|------|----------|
| `.names` | `array of str` | Atom names (`CA`, `N`, `CB`, …) |
| `.resnames` | `array of str` | Residue names (`ALA`, `GLY`, …) |
| `.resids` | `array of int` | Residue sequence numbers |
| `.masses` | `array of float` | Atomic masses (amu) |
| `.charges` | `array of float` | Partial charges (e) |
| `.positions` | `(N,3) float array` | XYZ coordinates in Å at the **current frame** |
| `.n_atoms` | `int` | Number of atoms in the group |

In [ ]:
# --- EXAMPLE ---
# Jump to the first frame and inspect Cα atom properties
u.trajectory[0]   # rewind to frame 0

print("Residue names:", ca_atoms.resnames)
print("Residue IDs  :", ca_atoms.resids)
print()
print("Cα positions at frame 0 (first 3 residues, Å):")
print(ca_atoms.positions[:3])
print()
print("Total mass of protein (amu):", protein.masses.sum().round(2))

#### ✏️ Practice 4.1
Using the `protein` AtomGroup:
- `total_mass` — total mass of the protein in amu
- `unique_resnames` — a sorted Python list of unique residue names (hint: `np.unique()` returns sorted unique values; convert to a list with `list()`)
- `n_heavy` — number of non-hydrogen protein atoms (reuse `heavy_protein` from above)

In [ ]:
# ✏️ YOUR CODE HERE
total_mass     = None   # amu
unique_resnames = None  # sorted list of residue name strings
n_heavy        = None   # integer count

raise NotImplementedError()

In [ ]:
# TEST 4.1
assert abs(total_mass - protein.masses.sum()) < 0.01, "total_mass is incorrect"
assert isinstance(unique_resnames, list), "unique_resnames should be a Python list"
assert unique_resnames == sorted(set(protein.resnames)), "unique_resnames should be sorted unique values"
assert n_heavy == heavy_protein.n_atoms, "n_heavy should equal heavy_protein.n_atoms"
print(f"✓  Mass = {total_mass:.1f} amu | Residues: {unique_resnames} | Heavy atoms: {n_heavy}")

---
## Section 5 — Iterating Over Frames

An MD trajectory is a sequence of **frames** — snapshots of atom positions separated by a fixed time interval (the trajectory write frequency). MDAnalysis exposes the trajectory as an iterable, so you can loop over it just like a Python list.

```python
for ts in u.trajectory:
    # ts is a Timestep object
    print(ts.frame, ts.time)   # frame index and time in ps
    # atom positions are updated automatically:
    pos = some_atom_group.positions
```

Each time the loop advances, MDAnalysis automatically updates the `.positions` of **all** `AtomGroup` objects so that they reflect the new frame — there is no need to pass the frame index explicitly.

You can also jump to a specific frame directly:
```python
u.trajectory[frame_index]
```

In [ ]:
# --- EXAMPLE ---
# Collect the time (ps) and the x-coordinate of the first Cα atom across all frames

times  = []
x_traj = []

for ts in u.trajectory:
    times.append(ts.time)                     # simulation time in ps
    x_traj.append(ca_atoms.positions[0, 0])   # x-coord of first Cα (Å)

times  = np.array(times)
x_traj = np.array(x_traj)

print(f"Frames collected: {len(times)}")
print(f"Time range: {times[0]:.1f} – {times[-1]:.1f} ps")
print(f"x range for first Cα: {x_traj.min():.2f} – {x_traj.max():.2f} Å")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, x_traj, linewidth=0.8)
plt.xlabel("Time (ns)")
plt.ylabel("x-coordinate of first Cα (Å)")
plt.title("First Cα x-position over time")
plt.tight_layout()
plt.show()

#### ✏️ Practice 5.1
Loop over the trajectory and build two arrays:
- `frame_times` — the simulation time (in **ns**) at each frame
- `n_protein_contacts` — for each frame, count the number of protein atoms within **4.0 Å** of the protein's own center of geometry (hint: use `protein.center_of_geometry()` to get the CoG as an xyz array, then use `u.select_atoms("protein and around 4.0 point X Y Z")` — see the cell below for the string formatting trick)

In [ ]:
# HINT: formatting an 'around point' selection string
# cog = protein.center_of_geometry()  # (x, y, z) array
# sel_string = f"protein and around 4.0 point {cog[0]:.3f} {cog[1]:.3f} {cog[2]:.3f}"

# ✏️ YOUR CODE HERE
frame_times       = []   # fill with time in ns at each frame
n_protein_contacts = []  # fill with atom count at each frame

for ts in u.trajectory:
    pass   # replace with your code

frame_times        = np.array(frame_times)
n_protein_contacts = np.array(n_protein_contacts)

raise NotImplementedError()

In [ ]:
# TEST 5.1
assert len(frame_times) == len(u.trajectory), "frame_times length mismatch"
assert np.allclose(frame_times[0], u.trajectory[0].time / 1000, atol=1e-6), "First time should be frame 0 time in ns"
assert len(n_protein_contacts) == len(u.trajectory), "n_protein_contacts length mismatch"
assert all(c >= 0 for c in n_protein_contacts), "Contact counts should be non-negative"
print("✓  Trajectory loop completed correctly")

plt.figure(figsize=(7, 3))
plt.plot(frame_times, n_protein_contacts, linewidth=0.8)
plt.xlabel("Time (ns)")
plt.ylabel("Atoms within 4 Å of CoG")
plt.title("Local packing density over time")
plt.tight_layout()
plt.show()

---
## Section 6 — Center of Mass and Radius of Gyration

Two simple but informative quantities for a peptide are:

**Center of mass (CoM):**  
The mass-weighted average position of all atoms, analogous to the balance point of a physical object.

$$\vec{r}_{\text{CoM}} = \frac{\sum_i m_i \vec{r}_i}{\sum_i m_i}$$

**Radius of gyration (R_g):**  
The root-mean-square distance of atoms from the CoM. A compact, folded peptide has a small R_g; an extended, unfolded one has a large R_g.

$$R_g = \sqrt{\frac{\sum_i m_i |\vec{r}_i - \vec{r}_{\text{CoM}}|^2}{\sum_i m_i}}$$

MDAnalysis provides both as single-line method calls on any `AtomGroup`:
```python
com = protein.center_of_mass()   # returns (x, y, z) in Å
rg  = protein.radius_of_gyration()  # returns scalar in Å
```

In [ ]:
# --- EXAMPLE ---
# Track radius of gyration over the trajectory

rg_values = []

for ts in u.trajectory:
    rg_values.append(protein.radius_of_gyration())

rg_values = np.array(rg_values)

print(f"Mean Rg   : {rg_values.mean():.2f} Å")
print(f"Std Rg    : {rg_values.std():.2f} Å")
print(f"Min Rg    : {rg_values.min():.2f} Å  (most compact)")
print(f"Max Rg    : {rg_values.max():.2f} Å  (most extended)")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, rg_values, linewidth=0.8, color='steelblue')
plt.axhline(rg_values.mean(), linestyle='--', color='red', label=f'Mean = {rg_values.mean():.2f} Å')
plt.xlabel("Time (ns)")
plt.ylabel("R$_g$ (Å)")
plt.title("Radius of Gyration over Time")
plt.legend()
plt.tight_layout()
plt.show()

#### ✏️ Practice 6.1
Compute the **end-to-end distance** of the peptide over the trajectory. The end-to-end distance is the Euclidean distance between the Cα of the **first** residue and the Cα of the **last** residue.

- `ca_first` — an `AtomGroup` containing the Cα of residue 1
- `ca_last` — an `AtomGroup` containing the Cα of the last residue (hint: `ca_atoms.resids[-1]`)
- `end_to_end` — a numpy array of the end-to-end distance (Å) at each frame

To compute the Euclidean distance between two position vectors `p1` and `p2`, use:  
```python
dist = np.linalg.norm(p1 - p2)
```

In [ ]:
# ✏️ YOUR CODE HERE
last_resid = ca_atoms.resids[-1]

ca_first   = None   # AtomGroup: Cα of residue 1
ca_last    = None   # AtomGroup: Cα of last residue
end_to_end = []     # distances at each frame

for ts in u.trajectory:
    pass   # replace with your code

end_to_end = np.array(end_to_end)

raise NotImplementedError()

In [ ]:
# TEST 6.1
assert ca_first.n_atoms == 1, "ca_first should contain exactly 1 atom"
assert ca_last.n_atoms == 1, "ca_last should contain exactly 1 atom"
assert ca_first.resids[0] == 1, "ca_first should be residue 1"
assert ca_last.resids[0] == ca_atoms.resids[-1], "ca_last should be the last residue"
assert len(end_to_end) == len(u.trajectory), "end_to_end should have one value per frame"
assert all(d >= 0 for d in end_to_end), "distances must be non-negative"
print(f"✓  End-to-end distance: {end_to_end.mean():.2f} ± {end_to_end.std():.2f} Å")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, end_to_end, linewidth=0.8, color='darkorange')
plt.xlabel("Time (ns)")
plt.ylabel("End-to-end distance (Å)")
plt.title("Peptide End-to-End Distance")
plt.tight_layout()
plt.show()

---
## Section 7 — RMSD: Measuring Structural Drift

**Root-Mean-Square Deviation (RMSD)** measures how much a structure has moved relative to a reference structure after optimal superposition. It is one of the most common quantities in MD analysis.

$$\text{RMSD}(t) = \sqrt{\frac{1}{N}\sum_{i=1}^{N} |\vec{r}_i(t) - \vec{r}_i^{\text{ref}}|^2}$$

where the sum is over the $N$ selected atoms and $\vec{r}_i^{\text{ref}}$ is the reference position of atom $i$ (usually the first frame or a known crystal structure).

**How to interpret RMSD:**
- An RMSD that quickly plateaus → the peptide reaches a stable conformation.
- An RMSD that keeps rising → the peptide is still drifting; the simulation may need to run longer.
- An RMSD that oscillates between two values → the peptide may be sampling two distinct states.

MDAnalysis provides `MDAnalysis.analysis.rms.RMSD` to handle the optimal rotation (superposition) automatically.

In [ ]:
# --- EXAMPLE ---
# Compute the backbone RMSD relative to the first frame
# MDAnalysis needs to align structures before measuring deviation,
# otherwise global translation/rotation would inflate the number.

R = rms.RMSD(
    u,                              # Universe to analyse
    select="backbone",              # atoms used for fitting AND RMSD
    ref_frame=0                     # reference: frame 0
)
R.run()

# Results are in R.results.rmsd: columns are [frame, time(ps), RMSD(Å)]
rmsd_data = R.results.rmsd
rmsd_time = rmsd_data[:, 1] / 1000   # convert ps → ns
rmsd_vals = rmsd_data[:, 2]           # RMSD in Å

print(f"Final RMSD  : {rmsd_vals[-1]:.2f} Å")
print(f"Maximum RMSD: {rmsd_vals.max():.2f} Å at {rmsd_time[rmsd_vals.argmax()]:.3f} ns")

plt.figure(figsize=(7, 3))
plt.plot(rmsd_time, rmsd_vals, linewidth=0.9, color='navy')
plt.xlabel("Time (ns)")
plt.ylabel("Backbone RMSD (Å)")
plt.title("RMSD relative to frame 0")
plt.tight_layout()
plt.show()

#### ✏️ Practice 7.1
Compute the RMSD of only the **Cα atoms** (instead of the full backbone) relative to frame 0. Store:
- `ca_rmsd_time` — time in **ns** for each frame
- `ca_rmsd_vals` — Cα RMSD in **Å** for each frame

Then print the mean Cα RMSD and plot the result.

In [ ]:
# ✏️ YOUR CODE HERE
R_ca = None   # rms.RMSD object

# don't forget to call .run()

ca_rmsd_time = None   # 1-D array, time in ns
ca_rmsd_vals = None   # 1-D array, RMSD in Å
mean_ca_rmsd = None   # scalar

raise NotImplementedError()

In [ ]:
# TEST 7.1
assert ca_rmsd_vals is not None, "ca_rmsd_vals not set"
assert len(ca_rmsd_vals) == len(u.trajectory), "ca_rmsd_vals length mismatch"
assert ca_rmsd_vals[0] < 1e-6, "RMSD at frame 0 (reference) must be 0"
assert abs(mean_ca_rmsd - ca_rmsd_vals.mean()) < 1e-6, "mean_ca_rmsd mismatch"
print(f"✓  Mean Cα RMSD: {mean_ca_rmsd:.2f} Å")

plt.figure(figsize=(7, 3))
plt.plot(ca_rmsd_time, ca_rmsd_vals, linewidth=0.9, color='teal')
plt.axhline(mean_ca_rmsd, linestyle='--', color='red', label=f'Mean = {mean_ca_rmsd:.2f} Å')
plt.xlabel("Time (ns)")
plt.ylabel("Cα RMSD (Å)")
plt.title("Cα RMSD relative to frame 0")
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 8 — RMSF: Per-Residue Flexibility

While RMSD tells you how much the whole structure moves over **time**, **Root-Mean-Square Fluctuation (RMSF)** tells you how much each residue fluctuates around its **time-averaged position**. It is a per-atom (or per-residue) property rather than a per-frame one.

$$\text{RMSF}_i = \sqrt{\left\langle |\vec{r}_i(t) - \langle\vec{r}_i\rangle|^2 \right\rangle_t}$$

Large RMSF → flexible loop or terminus.  
Small RMSF → rigid secondary structure (helix, sheet).

RMSF is closely related to the **B-factor** (also called the temperature factor) reported in PDB X-ray structures:

$$B = \frac{8\pi^2}{3}\,\text{RMSF}^2$$

This connection lets you compare simulation flexibility directly to crystallographic data.

In [ ]:
# --- EXAMPLE ---
# First align the trajectory to remove overall rotation/translation,
# then compute per-residue RMSF on the Cα atoms.

# Step 1: align all frames to the first frame (backbone fit)
aligner = align.AlignTraj(u, u, select="backbone", in_memory=True)
aligner.run()

# Step 2: compute RMSF on Cα atoms
rmsf_calc = rms.RMSF(ca_atoms)
rmsf_calc.run()

rmsf_values = rmsf_calc.results.rmsf  # one value per atom in ca_atoms

print("RMSF per Cα (Å):")
for resname, resid, r in zip(ca_atoms.resnames, ca_atoms.resids, rmsf_values):
    bar = '█' * int(r * 4)
    print(f"  {resname:3s} {resid:3d}: {r:5.2f} Å  {bar}")

plt.figure(figsize=(8, 3))
plt.bar(ca_atoms.resids, rmsf_values, color='mediumseagreen', edgecolor='none')
plt.xlabel("Residue number")
plt.ylabel("RMSF (Å)")
plt.title("Per-Residue Cα RMSF")
plt.tight_layout()
plt.show()

#### ✏️ Practice 8.1
The B-factor (crystallographic temperature factor) is related to RMSF by:

$$B_i = \frac{8\pi^2}{3}\,\text{RMSF}_i^2$$

Compute:
- `b_factors` — a numpy array of computed B-factors (Å²) for each Cα
- `most_flexible_resid` — the residue ID with the **largest** B-factor (hint: `np.argmax()` gives the index of the maximum; use that to index `ca_atoms.resids`)

In [ ]:
# ✏️ YOUR CODE HERE
b_factors           = None   # array of B-factors in Å²
most_flexible_resid = None   # integer residue ID

raise NotImplementedError()

In [ ]:
# TEST 8.1
expected_b = (8 * np.pi**2 / 3) * rmsf_values**2
assert np.allclose(b_factors, expected_b, atol=1e-8), "B-factor formula is off"
assert most_flexible_resid == ca_atoms.resids[np.argmax(rmsf_values)], "most_flexible_resid is incorrect"
print(f"✓  Most flexible residue: {ca_atoms.resnames[np.argmax(rmsf_values)]} {most_flexible_resid}")
print(f"   Its B-factor: {b_factors.max():.2f} Å²")

---
## Section 9 — Distances and Contacts

Computing distances between atoms is fundamental to structural biology. Common applications include:
- Checking whether a salt bridge forms and persists
- Monitoring hydrogen bond distances
- Counting native contacts to quantify folding

MDAnalysis offers two main approaches:

**1. Manual numpy distance** (fast for a single pair):
```python
d = np.linalg.norm(atom1.position - atom2.position)
```

**2. `MDAnalysis.lib.distances.dist()`** (handles periodic boundary conditions correctly, preferred for bulk properties):
```python
from MDAnalysis.lib import distances
d_array = distances.dist(group_a, group_b)   # returns (3, N) array: [frame_dist, ...]
```

For **contact maps** (is residue i close to residue j?), you can use `distances.self_distance_array()` on Cα positions.

In [ ]:
# --- EXAMPLE ---
# Monitor the distance between the Cα of residue 1 and residue N over the trajectory.
# This is the end-to-end Cα distance (complement to the end-to-end distance above).

from MDAnalysis.lib.distances import dist as mda_dist

ca_first_ag = u.select_atoms("name CA and resid 1")
ca_last_ag  = u.select_atoms(f"name CA and resid {ca_atoms.resids[-1]}")

dist_over_time = []
for ts in u.trajectory:
    result = mda_dist(ca_first_ag, ca_last_ag, box=ts.dimensions)
    dist_over_time.append(result[2][0])   # result[2] is the distance array

dist_over_time = np.array(dist_over_time)

print(f"Mean end-to-end (Cα) distance: {dist_over_time.mean():.2f} ± {dist_over_time.std():.2f} Å")

In [ ]:
# --- EXAMPLE: Contact map at the last frame ---
# A contact map shows which pairs of residues are in spatial proximity.
# Conventionally, two residues are 'in contact' if their Cα atoms are within 8 Å.

from MDAnalysis.lib.distances import self_distance_array

u.trajectory[-1]   # jump to last frame

# self_distance_array returns the upper-triangle of the pairwise distance matrix
ca_pos  = ca_atoms.positions
n_res   = len(ca_pos)
dists   = self_distance_array(ca_pos.astype(np.float32))

# Reconstruct full symmetric matrix from condensed upper-triangle
contact_matrix = np.zeros((n_res, n_res))
idx = np.triu_indices(n_res, k=1)
contact_matrix[idx]      = dists
contact_matrix           = contact_matrix + contact_matrix.T
contact_map              = contact_matrix < 8.0   # True if within 8 Å

plt.figure(figsize=(5, 4))
plt.imshow(contact_map, cmap='Blues', origin='lower',
           extent=[ca_atoms.resids[0], ca_atoms.resids[-1],
                   ca_atoms.resids[0], ca_atoms.resids[-1]])
plt.colorbar(label='Contact (< 8 Å)')
plt.xlabel("Residue i")
plt.ylabel("Residue j")
plt.title("Cα Contact Map (last frame)")
plt.tight_layout()
plt.show()

#### ✏️ Practice 9.1
Compute the **fraction of native contacts** over the trajectory. A native contact is defined as a pair of residues (|i − j| > 2) whose Cα atoms are within **8 Å** in the **first frame** (the reference). For each subsequent frame, count how many of those native contact pairs are still within 8 Å.

- `native_pairs` — a list of (i, j) index tuples where residues i and j are in contact at frame 0
- `q_values` — a numpy array giving the fraction Q(t) of native contacts retained at each frame

In [ ]:
# ✏️ YOUR CODE HERE

# Step 1: find native pairs at frame 0
u.trajectory[0]
native_pairs = []   # list of (i, j) index tuples with |i-j| > 2 and dist < 8 Å

# ... your code here ...

# Step 2: compute Q(t) over the trajectory
q_values = []   # one float per frame

for ts in u.trajectory:
    pass   # replace with your code

q_values = np.array(q_values)

raise NotImplementedError()

In [ ]:
# TEST 9.1
assert len(native_pairs) > 0, "No native pairs found — check your contact criterion"
assert all(abs(i - j) > 2 for i, j in native_pairs), "|i-j| > 2 not satisfied"
assert len(q_values) == len(u.trajectory), "q_values length mismatch"
assert q_values[0] == 1.0, "Q at frame 0 (native reference) should be 1.0"
assert all(0 <= q <= 1 for q in q_values), "Q values should be between 0 and 1"
print(f"✓  {len(native_pairs)} native contacts found, mean Q = {q_values.mean():.3f}")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, q_values, linewidth=0.9, color='purple')
plt.xlabel("Time (ns)")
plt.ylabel("Q (fraction native contacts)")
plt.title("Native Contact Fraction over Time")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Section 10 — Hydrogen Bond Analysis

Hydrogen bonds are critical to the secondary structure of peptides. An α-helix, for example, is stabilized by hydrogen bonds between the backbone N–H of residue i and the backbone C=O of residue i−4.

MDAnalysis defines a hydrogen bond by three geometric criteria:

| Criterion | Default threshold |
|-----------|-------------------|
| Donor–Acceptor distance | ≤ 3.5 Å |
| Donor–H–Acceptor angle | ≥ 120° |

The `HydrogenBondAnalysis` class automatically identifies donors and acceptors from atom names (O, N for acceptors; O–H, N–H for donors) and screens every frame of the trajectory.

> **Note on GROMACS .tpr files:** `.tpr` files do not always store explicit hydrogen atoms (some force fields use united-atom representations). If `HydrogenBondAnalysis` finds no H atoms, you may need to use a `.gro` or `.pdb` file as topology, or pass `hydrogens_sel` explicitly.

In [ ]:
# --- EXAMPLE ---
# Identify backbone hydrogen bonds within the protein

hbonds = HydrogenBondAnalysis(
    universe      = u,
    donors_sel    = "protein and name N",       # backbone N as donors
    hydrogens_sel = "protein and name H",       # backbone H
    acceptors_sel = "protein and name O",       # backbone O as acceptors
    d_a_cutoff    = 3.5,                        # donor–acceptor distance cutoff (Å)
    d_h_a_angle   = 120.0                       # minimum D–H···A angle (degrees)
)
hbonds.run()

# Count hydrogen bonds per frame
hb_count_per_frame = []
for frame_hbs in hbonds.results.hbonds:
    # frame_hbs is a list of [frame, donor_idx, H_idx, acceptor_idx, dist, angle]
    hb_count_per_frame.append(len(frame_hbs))

hb_count_per_frame = np.array(hb_count_per_frame)
print(f"Mean backbone H-bonds: {hb_count_per_frame.mean():.1f} ± {hb_count_per_frame.std():.1f}")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, hb_count_per_frame, linewidth=0.8, color='firebrick')
plt.xlabel("Time (ns)")
plt.ylabel("Number of backbone H-bonds")
plt.title("Backbone Hydrogen Bonds over Time")
plt.tight_layout()
plt.show()

#### ✏️ Practice 10.1
Using the `hbonds` results, find the **most persistent** backbone hydrogen bond — the donor–acceptor pair that appears in the largest number of frames.

- `all_donor_idx` — a flat numpy array of donor atom indices from every H-bond event
- `all_accept_idx` — a flat numpy array of acceptor atom indices from every H-bond event
- `top_donor_resid` — the residue ID of the most common donor atom
- `top_accept_resid` — the residue ID of the most common corresponding acceptor atom

Hint: `hbonds.results.hbonds` is a list of lists; each inner list contains `[frame, donor_idx, H_idx, acceptor_idx, dist, angle]`. Use `np.concatenate` to flatten them after collecting.

For the most common pair, you can form (donor_idx, acceptor_idx) tuples and use `np.unique` with `return_counts=True` to count occurrences.

In [ ]:
# ✏️ YOUR CODE HERE
all_donor_idx   = None
all_accept_idx  = None
top_donor_resid  = None
top_accept_resid = None

raise NotImplementedError()

In [ ]:
# TEST 10.1
assert all_donor_idx is not None and len(all_donor_idx) > 0, "all_donor_idx is empty"
assert len(all_donor_idx) == len(all_accept_idx), "donor/acceptor arrays must have equal length"
# Verify the top pair is a valid protein residue
assert top_donor_resid in ca_atoms.resids, "top_donor_resid not in protein residue list"
assert top_accept_resid in ca_atoms.resids, "top_accept_resid not in protein residue list"
print(f"✓  Most persistent H-bond: residue {top_donor_resid} (donor N) → residue {top_accept_resid} (acceptor O)")

---
## Section 11 — A Reusable Analysis Script

Throughout this notebook you have written individual analysis snippets. In practice, you will want to package these into reusable functions and run them from the command line. The cell below shows how to organise all the analyses you have learned into a single, well-documented Python function.

This section is **read-and-run** — study the structure and then customise it for your own project.

In [ ]:
# --- Complete reusable analysis function ---

def analyze_peptide(tpr_path: str, xtc_path: str, outdir: str = ".") -> dict:
    """
    Run a standard set of analyses on a GROMACS peptide simulation.

    Parameters
    ----------
    tpr_path : str
        Path to the GROMACS .tpr topology file.
    xtc_path : str
        Path to the GROMACS .xtc trajectory file.
    outdir : str
        Directory where result plots will be saved.

    Returns
    -------
    dict
        Dictionary containing the computed arrays:
        'time_ns', 'rg', 'rmsd', 'rmsf', 'hbond_count'
    """
    import os
    os.makedirs(outdir, exist_ok=True)

    # --- Load universe ---
    u = mda.Universe(tpr_path, xtc_path)
    protein   = u.select_atoms("protein")
    backbone  = u.select_atoms("backbone")
    ca_atoms  = u.select_atoms("protein and name CA")

    # --- Time axis ---
    times_ns = np.array([ts.time / 1000 for ts in u.trajectory])

    # --- Radius of gyration ---
    rg_vals = []
    for ts in u.trajectory:
        rg_vals.append(protein.radius_of_gyration())
    rg_vals = np.array(rg_vals)

    # --- RMSD ---
    R = rms.RMSD(u, select="backbone", ref_frame=0)
    R.run()
    rmsd_vals = R.results.rmsd[:, 2]

    # --- RMSF (after alignment) ---
    aligner = align.AlignTraj(u, u, select="backbone", in_memory=True)
    aligner.run()
    rmsf_calc = rms.RMSF(ca_atoms)
    rmsf_calc.run()
    rmsf_vals = rmsf_calc.results.rmsf

    # --- Hydrogen bonds ---
    hbonds = HydrogenBondAnalysis(
        universe=u,
        donors_sel="protein and name N",
        hydrogens_sel="protein and name H",
        acceptors_sel="protein and name O",
    )
    hbonds.run()
    hb_counts = np.array([len(f) for f in hbonds.results.hbonds])

    # --- Plots ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(times_ns, rg_vals, linewidth=0.9, color='steelblue')
    axes[0, 0].set(xlabel='Time (ns)', ylabel='Rg (Å)', title='Radius of Gyration')

    axes[0, 1].plot(times_ns, rmsd_vals, linewidth=0.9, color='navy')
    axes[0, 1].set(xlabel='Time (ns)', ylabel='RMSD (Å)', title='Backbone RMSD')

    axes[1, 0].bar(ca_atoms.resids, rmsf_vals, color='mediumseagreen', edgecolor='none')
    axes[1, 0].set(xlabel='Residue', ylabel='RMSF (Å)', title='Per-Residue RMSF')

    axes[1, 1].plot(times_ns, hb_counts, linewidth=0.8, color='firebrick')
    axes[1, 1].set(xlabel='Time (ns)', ylabel='H-bonds', title='Backbone H-Bonds')

    plt.tight_layout()
    fig.savefig(f"{outdir}/peptide_summary.png", dpi=150)
    plt.show()
    print(f"Summary figure saved to {outdir}/peptide_summary.png")

    return {
        'time_ns'     : times_ns,
        'rg'          : rg_vals,
        'rmsd'        : rmsd_vals,
        'rmsf'        : rmsf_vals,
        'hbond_count' : hb_counts,
    }


# --- Run the full analysis ---
results = analyze_peptide(TPR, XTC, outdir="peptide_analysis_output")

print("\n=== Summary ===")
print(f"Mean Rg    : {results['rg'].mean():.2f} Å")
print(f"Mean RMSD  : {results['rmsd'].mean():.2f} Å")
print(f"Mean H-bonds: {results['hbond_count'].mean():.1f}")

---
## Summary and Next Steps

You have worked through a complete MDAnalysis workflow for peptide simulations:

1. **Loading** a `Universe` from GROMACS `.tpr` + `.xtc` files
2. **Selecting** atom subsets with human-readable selection strings
3. **Inspecting** atom attributes (names, masses, positions)
4. **Iterating** over trajectory frames
5. **Computing** global structural properties (Rg, end-to-end distance)
6. **Measuring** structural drift with RMSD
7. **Quantifying** flexibility with RMSF and B-factors
8. **Analysing** pairwise distances and native contacts
9. **Identifying** hydrogen bonds and their persistence
10. **Packaging** everything into a reusable function

**Where to go next:**
- **Dihedral angles (Ramachandran plots):** `MDAnalysis.analysis.dihedrals`
- **Secondary structure:** use `MDAnalysis` with `mdtraj` or DSSP for per-frame secondary structure
- **Solvent-accessible surface area (SASA):** `MDAnalysis.analysis.hydrophobicanalysis` or Freesasa
- **Water structure:** analyse solvent shells around the peptide
- **Dimensionality reduction (PCA/TICA):** `MDAnalysis.analysis.pca` for identifying dominant motions

**Useful references:**
- MDAnalysis documentation: https://docs.mdanalysis.org
- MDAnalysis User Guide: https://userguide.mdanalysis.org
- MDAnalysis publication: Michaud-Agrawal et al., *J. Comput. Chem.* 2011, 32, 2319–2327